<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit3/session-14-deploy-and-operate/notebook.ipynb)


# Session 14 — Deploy and operate the capstone

**Goal:** put the capstone behind a request boundary, then write the smoke test that can tell you the deployment is bad. *Thread: harness engineering.*

Every cell runs offline. No network, no key, no hosting account. The deployment you smoke-test here is a process on your own machine and four fakes.

In [ ]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
CORPUS_DIR = REPO_ROOT / "data" / "corpus"

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    if "google.colab" in sys.modules:
        # Colab starts in /content with no course in it, so fetch one. A shallow
        # clone of the COHORT repository, which is the public one; the source
        # repository is private and would ask this learner for credentials.
        import subprocess

        target = Path("/content/dev3pack")
        if not (target / "pyproject.toml").exists():
            print("Colab detected — fetching the course (about 20 seconds)…")
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1",
                 "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(target)],
                check=True,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(target)], check=True
        )
        REPO_ROOT = target
        sys.path.insert(0, str(REPO_ROOT / "src"))
        import os

        os.chdir(REPO_ROOT)
        from bootcamp_agent.preflight import preflight

        print(f"ready — the course is at {REPO_ROOT}")
    else:
        print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
        print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    LIVE = preflight(REPO_ROOT)  # the configured lane's client; FakeLLM whenever the lane is down

In [ ]:
from bootcamp_agent.checks import check, review

## 1. The deployment, at its smallest honest size

A deployment is your code in a process you are not sitting in front of, reachable over a boundary somebody else can call. The boundary is the part that matters, and you can have it today: one route that answers, one that says whether the process is alive, and a body that either validates or is refused.

`local_service` below is the capstone with that boundary drawn around it. Run it in one process now; put it behind `uvicorn`, a container or a platform later. Nothing downstream of here cares which.

It reports `elapsed_ms: 0`, because nothing here starts a process. That zero is the part a real deployment takes away from you, and section 2 is where you meet it.

In [ ]:
import time

from bootcamp_agent.agent import answer_question
from bootcamp_agent.documents import load_corpus
from bootcamp_agent.llm import FakeLLM

documents = load_corpus(CORPUS_DIR)


def local_service(path: str, body: dict | None = None) -> dict:
    """Your capstone behind a request boundary: same process, no network, service shape."""
    started = time.perf_counter()

    def done(status: int, payload: dict) -> dict:
        elapsed = round((time.perf_counter() - started) * 1000)
        return {"status": status, "body": payload, "elapsed_ms": elapsed}

    if path == "/health":
        return done(200, {"ok": True, "documents": len(documents)})
    if path != "/answer":
        return done(404, {"error": f"no route {path!r}"})
    if not (isinstance(body, dict) and isinstance(body.get("question"), str)):
        return done(400, {"error": "'question' must be a non-empty string"})
    answer = answer_question(body["question"], documents, FakeLLM()).answer
    return done(200, {
        "answer": answer.answer,
        "citations": list(answer.citations),
        "confidence": answer.confidence,
        "needs_human_review": answer.needs_human_review,
    })


print("health   ", local_service("/health"))
print("answer   ", local_service("/answer", {"question": "How does chunking work?"})["status"])
print("malformed", local_service("/answer", {"quesiton": 12}))

## 2. Four deployments, four ways to be wrong

The same call shape, four behaviours you will meet the week you ship:

| kind | what it does | the failure it stands for |
|---|---|---|
| `warm` | up, fast, strict | none — this is the one you hope for |
| `cold` | 1,900 ms on the first call, 25 ms after | the cold start |
| `lax` | 200 and an invented answer for a malformed body | the boundary that stopped validating |
| `killed` | 503, container over its memory limit | the resource limit |

The delay is a number in the response, not a wait, so every cell here is instant and identical on every machine.

In [ ]:
ANSWER = {
    "answer": "Chunking splits a document into passages.",
    "citations": ["rag-basics"],
    "confidence": 0.82,
    "needs_human_review": False,
}


def fake_deployment(kind: str):
    """Four deployments behind one call shape. Nothing sleeps; the delay is a number."""
    cold, warm = {"warm": (30, 30), "cold": (1900, 25), "lax": (70, 70), "killed": (110, 110)}[kind]
    seen = []

    def request(path: str, body: dict | None = None) -> dict:
        seen.append(path)
        elapsed = cold if len(seen) == 1 else warm
        if kind == "killed":
            error = {"error": "container killed: memory limit exceeded"}
            return {"status": 503, "body": error, "elapsed_ms": elapsed}
        if path == "/health":
            return {"status": 200, "body": {"ok": True}, "elapsed_ms": elapsed}
        if isinstance(body, dict) and isinstance(body.get("question"), str):
            return {"status": 200, "body": ANSWER, "elapsed_ms": elapsed}
        if kind == "lax":
            invented = {**ANSWER, "answer": "Yes, that is correct.", "citations": [],
                        "confidence": 0.9}
            return {"status": 200, "body": invented, "elapsed_ms": elapsed}
        return {"status": 400, "body": {"error": "'question' must be a non-empty string"},
                "elapsed_ms": elapsed}

    return request


for kind in ("warm", "cold", "lax", "killed"):
    service = fake_deployment(kind)
    health = service("/health")
    garbage = service("/answer", {"quesiton": 12})
    print(f"{kind:7} first call {health['elapsed_ms']:>5} ms  "
          f"health {health['status']}  malformed -> {garbage['status']} {garbage['body']}")

## 3. Exercise: the deployment smoke test

**Context.** A smoke test that only passes when everything is fine tells you nothing. This one has to be able to come back and say the deployment is bad — and it has to survive the deployment being bad, because a smoke test that crashes on the broken one reports nothing exactly when the report was the point.

`request(path, body=None)` is handed to you. It returns `{"status": int, "body": dict, "elapsed_ms": int}` and it never raises. `elapsed_ms` is what the service reports; a wall clock reads zero here, because nothing really sleeps.

**The report contract.** Four fields, and the check holds you to each one.

| Field | True when |
|---|---|
| `cold_start_ms` | the `elapsed_ms` of your FIRST call, as an int |
| `malformed_rejected` | the malformed body came back 4xx. A 200 is a `false`, not a pass |
| `healthy` | `/health` answered 200, the good question answered 200 with all four answer fields, and the malformed body was refused |
| `rollback` | one sentence naming the way back: an action, a number, a unit |

**Instructions.**

1. Send the same three probes to every deployment, whatever the first one answers: `/health`, one good question, one malformed body.
2. Read `status`, never the presence of a body. `killed` returns a body too.
3. Fill the four fields from what came back. Report bad news as bad news.
4. Write `rollback` now, before anything is broken — the moment you need it is the moment you cannot think. Name the action, and put a number with a unit in it, the same standard an architecture decision's reversal trigger has to meet.
5. Run the check. It drives your function with four deployments of its own.

In [ ]:
QUESTION = "How does chunking work in retrieval-augmented generation?"
MALFORMED = {"quesiton": 12}  # a typo in the field name, and an int where a string goes


def smoke(request) -> dict:
    """Exercise the deployment and report what it did — good news or bad."""
    health = request("/health")  # the first call, and its elapsed_ms is the cold start
    # TODO(you) 1: send two more probes to "/answer" — one QUESTION, one MALFORMED body
    # TODO(you) 2: fill the report from the three responses. A 200 for garbage is not
    #              a rejection, and a field you did not probe is a claim, not a result.
    return {
        "cold_start_ms": 0,
        "malformed_rejected": False,
        "healthy": False,
        "rollback": "",  # TODO(you) 3: an action, a number and a unit
    }


for kind in ("warm", "cold", "lax", "killed"):
    print(f"{kind:7}", smoke(fake_deployment(kind)))

**Expected output** (your rollback sentence will differ; the verdicts will not):

```
warm    {'cold_start_ms': 30, 'malformed_rejected': True, 'healthy': True, ...}
cold    {'cold_start_ms': 1900, 'malformed_rejected': True, 'healthy': True, ...}
lax     {'cold_start_ms': 70, 'malformed_rejected': False, 'healthy': False, ...}
killed  {'cold_start_ms': 110, 'malformed_rejected': False, 'healthy': False, ...}
✅ ch14-e3 passed
```

`cold` is healthy and slow. Those are two different facts and the report keeps them apart: one number a reader judges against a budget, one verdict about the contract.

In [ ]:
check("ch14-e3", smoke)

## 4. Point it at the service you actually run

Same function, your own process. This is the required path: a service you can start, a smoke test you can run against it, and a report you can read.

In [ ]:
print(smoke(local_service))

**Optional, and local.** Serve the capstone on a port in another terminal, set `SMOKE_URL`, and rerun the cell below. The transport changes; `smoke` does not. With `SMOKE_URL` unset the cell skips and nothing is fetched.

In [ ]:
import os

url = os.environ.get("SMOKE_URL")  # e.g. http://127.0.0.1:8000
if not url:
    print("SMOKE_URL is not set — skipped, and nothing was fetched.")
    print("Serve your capstone on a port, then: SMOKE_URL=http://127.0.0.1:8000 and rerun.")
else:
    import json
    import time
    import urllib.error
    import urllib.request

    def http_request(path: str, body: dict | None = None) -> dict:
        """The same call shape over HTTP. Only the transport changed."""
        data = None if body is None else json.dumps(body).encode()
        headers = {"Content-Type": "application/json"}
        call = urllib.request.Request(url + path, data=data, headers=headers)
        started = time.perf_counter()
        try:
            with urllib.request.urlopen(call, timeout=10) as response:
                status, raw = response.status, response.read()
        except urllib.error.HTTPError as error:  # a 4xx is an answer, not an outage
            status, raw = error.code, error.read()
        except OSError as error:  # nothing listening: report it, do not raise
            status, raw = 0, json.dumps({"error": str(error)}).encode()
        elapsed = round((time.perf_counter() - started) * 1000)
        try:
            payload = json.loads(raw or b"{}")
        except json.JSONDecodeError:
            payload = {"error": "the service did not return JSON"}
        return {"status": status, "body": payload, "elapsed_ms": elapsed}

    print(smoke(http_request))

## 5. Failure injection: the smoke test that always passes

`rosy` is the first smoke test everybody writes. It asks whether the service answered.

In [ ]:
def rosy(request) -> dict:
    """The smoke test everybody writes first: it asks whether anything answered."""
    request("/health")
    good = request("/answer", {"question": QUESTION})
    return {
        "cold_start_ms": 0,
        "malformed_rejected": True,
        "healthy": good["status"] == 200,
        "rollback": "",
    }


print("rosy on the lax deployment: ", rosy(fake_deployment("lax")))
print("yours on the lax deployment:", smoke(fake_deployment("lax")))
print()
print("The lax deployment answered 'Yes, that is correct.' to a body with a typo for a")
print("field name, status 200, confidence 0.9, no citations. rosy never sent one, so it")
print("reports a healthy deployment. It is not measuring anything; it is agreeing.")

## Exit ticket

One thing that works, one thing that is unclear, your next action.

Homework: run `smoke` against your own capstone, write the rollback sentence into your README, and bring tomorrow the one deployment failure you cannot yet handle.

## Review

The scorecard for this notebook. Every ❌ line names the exercise and the hint.

In [ ]:
review("ch14")